# Diagnóstico para el Observatorio CMF Chile

Este notebook revisa tus archivos DEIS, GRD, INE, nacidos vivos e IPC **en tu computador** y exporta solo
**tablas agregadas** (conteos por año, etiquetas de variables, nombres de columnas). No exporta registros
individuales, identificadores de pacientes ni fechas.

**Objetivo:** que el observatorio reproduzca exactamente el método de tus publicaciones. Para eso necesito confirmar:

1. Cómo vienen los tramos de `GRUPO_EDAD` (DEIS) y `TRAMO_EDAD` (GRD 2025).
2. Si los códigos CIE-10 traen punto, y cómo vienen comuna, región, sexo, previsión, condición de egreso, `TIPOALTA`, `TIPO_ACTIVIDAD` y `USOSPABELLON`.
3. Cuántos casos da cada **definición candidata** por año (trauma con y sin S02.0, etc.), para compararlos con las cifras publicadas.
4. La estructura del archivo INE comunal, del archivo de nacidos vivos y del IPC.
5. *(Opcional)* Que el pipeline del observatorio corra con tus archivos y reproduzca tus series.

**Uso:** edita solo la celda de configuración y ejecuta **Restart Kernel and Run All**. Al final queda un
`diagnostico_observatorio_<fecha>.zip`. Mándame ese zip y responde las preguntas de la última sección.

Requisitos: `duckdb`, `pandas`, `openpyxl` (los mismos de tus notebooks). La sección 5 requiere además R con
`data.table`, `yaml`, `jsonlite`, `digest` y `readxl`.

## 0. Configuración

In [ ]:
from pathlib import Path

# Carpetas y archivos (mismas rutas de tus notebooks)
DEIS_DIR     = Path(r"C:/Users/gusta/Downloads/Estudio tendencias fx maxilo/data_raw")
DEIS_PATTERN = "EGRE_DATOS_ABIERTOS_{year}.csv"
GRD_DIR      = Path(r"C:/Users/gusta/Downloads/economic burden del trauma")
GRD_PATTERN  = "GRD_PUBLICO_{year}.txt"
INE_FILE     = GRD_DIR / "estimaciones-y-proyecciones-2002-2035-comuna-y-área-urbana-y-rural.xlsx"
INE_REGION_FILE = Path(r"C:/Users/gusta/Downloads/Estudio tendencias fx maxilo/ine_region.xlsx")  # libro regional (infecciones)
BIRTHS_REGION_FILE = Path(r"C:/Users/gusta/Downloads/Estudio FLP/nacimientos_region_2001_2023.csv")
BIRTHS_FILE  = Path(r"C:/Users/gusta/Downloads/Estudio FLP/nacimientos_chile.csv")
DENOM_FLP_FILE = Path(r"C:/Users/gusta/Downloads/Estudio FLP/denominador_flp.xlsx")
CPI_FILE     = GRD_DIR / "Indicador.xls"
PROD_FILE    = GRD_DIR / "parametros_productividad_2019_2025.csv"

DEIS_YEARS = list(range(2001, 2026))   # se omiten los años sin archivo
GRD_YEARS  = list(range(2019, 2026))

# Salidas agregadas que ya generaron tus notebooks (se copian al zip si existen; sirven para cuadrar cifras)
OUTPUTS_PREVIOS = [
    GRD_DIR / "output_2019_2025" / "control_cohorte_por_anio.csv",
    Path(r"C:/Users/gusta/Downloads/Estudio FLP/outputs_craneosinostosis_GRD_v3/tablas/episodios_por_definicion.csv"),
    Path(r"C:/Users/gusta/Downloads/Estudio FLP/outputs_craneosinostosis_series/tablas/serie_nacional.csv"),
    Path(r"C:/Users/gusta/Downloads/Estudio FLP/outputs_craneosinostosis_series/tablas/volumenes_para_GRD.csv"),
    Path(r"C:/Users/gusta/Downloads/Estudio FLP/outputs_geo_revisado/tablas/tabla1_tipo_fisura_menores_1.csv"),
    Path(r"C:/Users/gusta/Downloads/Estudio FLP/outputs_geo_revisado/tablas/denominador_ine_vs_nacidos_vivos.csv"),
]

# Sección 5 (opcional): correr el pipeline del observatorio con tus archivos
RUN_PIPELINE = False
REPO_DIR = Path(r"C:/Users/gusta/Downloads/observatoriocmf")   # carpeta descomprimida del zip del repositorio
RSCRIPT = None   # p. ej. r"C:\Program Files\R\R-4.4.1\bin\Rscript.exe"; None = detección automática
PIPE_DEIS_YEARS = list(range(2001, 2025))
PIPE_GRD_YEARS = list(range(2019, 2026))

OUT = Path.cwd() / "diagnostico_observatorio"

In [ ]:
import datetime, json, re, shutil, unicodedata, zipfile
import duckdb
import pandas as pd

OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
con = duckdb.connect()
LOG = []

def log(msg):
    print(msg); LOG.append(str(msg))

def save(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8")
    log(f"  → {name}.csv ({len(df):,} filas)")

def norm_col(c):
    c = str(c).replace("\ufeff", "").replace("ï»¿", "").strip()
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Z0-9]+", "_", c.upper()).strip("_")

def qi(c):  # identificador SQL entre comillas
    return '"' + str(c).replace('"', '""') + '"'

def csv_src(path, delim, enc, sample=-1):
    p = str(path).replace("\\", "/").replace("'", "''")
    return (f"read_csv('{p}', delim='{delim}', header=true, all_varchar=true, encoding='{enc}', "
            f"strict_mode=false, ignore_errors=true, null_padding=true, max_line_size=10000000, sample_size={sample})")

def sniff_enc(path):
    """UTF-16 si trae BOM (GRD 2022 y 2023); UTF-8 si los primeros 200 KB decodifican; si no, Latin-1."""
    raw = Path(path).open("rb").read(200_000)
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return ["utf-16"]
    try:
        raw.decode("utf-8"); return ["utf-8", "latin-1"]
    except UnicodeDecodeError:
        return ["latin-1", "utf-8"]

LOAD_INFO = {}

def header_line(path, enc):
    with Path(path).open("rb") as f:
        return f.readline(500_000).decode(enc, errors="replace").strip("\r\n\ufeff")

def sniff_delim(path, enc, preferred):
    """El separador indicado, salvo que el encabezado tenga claramente otro (p. ej. un año GRD con ';')."""
    line = header_line(path, enc)
    counts = {d: line.count(d) for d in dict.fromkeys([preferred, "|", ";", "\t", ","])}
    best = max(counts, key=counts.get)
    return preferred if counts[preferred] >= max(1, counts[best]) else best

def utf16_to_utf8(path):
    """DuckDB no lee UTF-16: copia temporal en UTF-8 (por bloques; se borra después de cargar la tabla)."""
    tmp = Path(OUT).parent / "_tmp_utf8"; tmp.mkdir(exist_ok=True)
    dest = tmp / (Path(path).stem + "_utf8" + Path(path).suffix)
    log(f"{Path(path).name}: UTF-16 → UTF-8 temporal ({dest})")
    with open(path, "r", encoding="utf-16", newline="") as fi, open(dest, "w", encoding="utf-8", newline="") as fo:
        while True:
            chunk = fi.read(16 * 1024 * 1024)
            if not chunk:
                break
            fo.write(chunk)
    return dest

def load_table(path, delim, wanted=None, patterns=(), name="t"):
    """Materializa en DuckDB solo las columnas necesarias, con nombres normalizados."""
    if sniff_enc(path) == ["utf-16"]:
        conv = utf16_to_utf8(path)
        try:
            enc, cols = load_table(conv, delim, wanted, patterns, name)
        finally:
            conv.unlink(missing_ok=True)
        return "utf-16", cols
    last = None
    for enc in sniff_enc(path):
        try:
            d = sniff_delim(path, enc, delim)
            src = csv_src(path, d, enc)
            cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {csv_src(path, d, enc, 20480)}").fetchall()]
            m = {}
            for c in cols:
                n = norm_col(c)
                if n in m.values():
                    continue
                if wanted is None or n in wanted or any(re.fullmatch(p, n) for p in patterns):
                    m[c] = n
            if not m:
                raise ValueError(f"{Path(path).name}: ninguna columna esperada con separador {d!r} y codificación "
                                 f"{enc}. Encabezado (solo nombres): {header_line(path, enc)[:400]!r}")
            sel = ", ".join(f"{qi(c)} AS {qi(n)}" for c, n in m.items())
            con.execute(f"CREATE OR REPLACE TEMP TABLE {name} AS SELECT {sel} FROM {src}")
            LOAD_INFO[name] = {"separador": d, "encabezado": header_line(path, enc)[:400]}
            return enc, [norm_col(c) for c in cols]
        except Exception as e:  # noqa: BLE001
            last = e
    raise last

def has(col, table="t"):
    return col in [r[0] for r in con.execute(f"DESCRIBE {table}").fetchall()]

def value_counts(table, col, anio, extra=""):
    df = con.execute(f"SELECT {qi(col)} AS valor, count(*) AS n FROM {table} {extra} GROUP BY 1 ORDER BY 2 DESC").df()
    df.insert(0, "columna", col); df.insert(0, "anio", anio)
    return df

def shape_expr(col):  # forma del valor sin revelarlo: dígitos → 9, letras → A
    return f"regexp_replace(regexp_replace(coalesce({qi(col)}, '<NA>'), '[0-9]', '9', 'g'), '[A-Za-z]', 'A', 'g')"

log(f"duckdb {duckdb.__version__}, pandas {pd.__version__}, {datetime.datetime.now():%Y-%m-%d %H:%M}")

## 1. DEIS: estructura y etiquetas

Por año: columnas presentes, codificación, filas, valores de las variables categóricas (incluido `GRUPO_EDAD`),
formato de los códigos CIE-10 y de la comuna de residencia.

In [ ]:
DEIS_CAT = ["GRUPO_EDAD", "SEXO", "PREVISION", "PERTENENCIA_ESTABLECIMIENTO_SALUD", "CONDICION_EGRESO", "INTERV_Q",
            "REGION_RESIDENCIA", "TIPO_EDAD", "EDAD_TIPO", "SERVICIO_SALUD", "SEREMI_SERVICIO_SALUD"]
DEIS_KEEP = set(DEIS_CAT) | {"ANO_EGRESO", "DIAG1", "DIAG2", "COMUNA_RESIDENCIA", "DIAS_ESTADA", "EDAD", "EDAD_CANT",
                             "EDAD_ANIOS", "GLOSA_PREVISION"}
deis_years = [y for y in DEIS_YEARS if (DEIS_DIR / DEIS_PATTERN.format(year=y)).exists()]
log(f"Años DEIS encontrados: {deis_years}")

cols_rows, vc_rows, fmt_rows, com_rows, info_rows = [], [], [], [], []
for y in deis_years:
    enc, allcols = load_table(DEIS_DIR / DEIS_PATTERN.format(year=y), ";", DEIS_KEEP, name="deis")
    n = con.execute("SELECT count(*) FROM deis").fetchone()[0]
    info_rows.append({"anio": y, "encoding": enc, "filas": n, "n_columnas": len(allcols)})
    cols_rows += [{"anio": y, "columna": c} for c in allcols]
    for c in DEIS_CAT:
        if has(c, "deis"):
            vc_rows.append(value_counts("deis", c, y))
    for c in ["EDAD", "EDAD_CANT", "EDAD_ANIOS", "DIAS_ESTADA", "ANO_EGRESO"]:
        if has(c, "deis"):
            vc_rows.append(value_counts("deis", c, y).head(150))
    for c in ["DIAG1", "DIAG2"]:
        if has(c, "deis"):
            f = con.execute(f"""SELECT {shape_expr(c)} AS forma, count(*) AS n FROM deis GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").df()
            f.insert(0, "columna", c); f.insert(0, "anio", y); fmt_rows.append(f)
    if has("COMUNA_RESIDENCIA", "deis"):
        c = con.execute(f"""SELECT {shape_expr('COMUNA_RESIDENCIA')} AS forma, count(*) AS n FROM deis GROUP BY 1 ORDER BY 2 DESC""").df()
        c.insert(0, "anio", y); com_rows.append(c)
        txt = con.execute("""SELECT COMUNA_RESIDENCIA AS valor, count(*) AS n FROM deis
                             WHERE NOT regexp_matches(coalesce(COMUNA_RESIDENCIA,''), '^[0-9]+(\\.0+)?$')
                             GROUP BY 1 ORDER BY 2 DESC LIMIT 400""").df()
        txt.insert(0, "columna", "COMUNA_RESIDENCIA_texto"); txt.insert(0, "anio", y); vc_rows.append(txt)
        cr = con.execute("""SELECT REGION_RESIDENCIA AS region, substr(lpad(regexp_replace(COMUNA_RESIDENCIA, '\\.0+$', ''), 5, '0'), 1, 2) AS comuna_2dig,
                                   count(*) AS n FROM deis
                            WHERE regexp_matches(coalesce(COMUNA_RESIDENCIA,''), '^[0-9]+(\\.0+)?$') GROUP BY 1, 2""").df() if has("REGION_RESIDENCIA", "deis") else pd.DataFrame()
        if len(cr) and y in (deis_years[0], 2007, 2008, 2018, 2019, deis_years[-1]):
            cr.insert(0, "anio", y); save(cr, f"deis_region_vs_comuna_{y}")
    log(f"[{y}] enc={enc} filas={n:,}")

save(pd.DataFrame(info_rows), "deis_info_archivos")
save(pd.DataFrame(cols_rows), "deis_columnas")
save(pd.concat(vc_rows, ignore_index=True), "deis_valores")
save(pd.concat(fmt_rows, ignore_index=True), "deis_formato_cie")
if com_rows: save(pd.concat(com_rows, ignore_index=True), "deis_formato_comuna")

ge = pd.concat(vc_rows, ignore_index=True).query("columna == 'GRUPO_EDAD'")
print("\nEtiquetas de GRUPO_EDAD (unión de años):")
print(ge.groupby("valor")["n"].sum().sort_values(ascending=False).to_string())

## 2. DEIS: definiciones candidatas por año

Cuenta los casos de cada definición usada en tus scripts, con el diagnóstico principal tal como lo leían
(`substr(DIAG1, 1, 4)`, sin quitar puntos) y también con los puntos removidos, por si difieren.

| Definición | Códigos (4 caracteres) | Origen |
|---|---|---|
| `trauma_A_blando_duro` | S00.1, S00.3-5, S01.2-5 + S02.2-4, S02.6-9 | `Este_si.R` |
| `trauma_B_con_S020` | A + S02.0 | `final_final.R` |
| `trauma_C_facial` | A + S03.0, S03.1, S03.4, S05.4, S07.0 | notebook 2019-2025 |
| `trauma_D_amplio` | C + S02.0, S02.1 | CMFT amplio (sensibilidad) |
| `universo_trauma` | S00-S99 y T00-T14 | denominador por 1.000 egresos por trauma |

In [ ]:
SOFT = ["S001", "S003", "S004", "S005", "S012", "S013", "S014", "S015"]
HARD = ["S022", "S023", "S024", "S026", "S027", "S028", "S029"]
OTHER = ["S030", "S031", "S034", "S054", "S070"]
DEFS = {"trauma_A_blando_duro": SOFT + HARD, "trauma_B_con_S020": SOFT + HARD + ["S020"],
        "trauma_C_facial": SOFT + HARD + OTHER, "trauma_D_amplio": SOFT + HARD + OTHER + ["S020", "S021"],
        "trauma_S020": ["S020"], "trauma_S021": ["S021"]}
INF4 = ["K102", "K112", "K113", "K122", "L020", "L021", "L032", "J390", "J391", "H050"]
lst = lambda xs: ", ".join(f"'{x}'" for x in xs)

def def_counts(expr4, etiqueta):
    parts = [f"sum(CASE WHEN {expr4} IN ({lst(v)}) THEN 1 ELSE 0 END) AS {k}" for k, v in DEFS.items()]
    parts.append(f"sum(CASE WHEN ({expr4} BETWEEN 'S000' AND 'S999') OR ({expr4} BETWEEN 'T000' AND 'T149') THEN 1 ELSE 0 END) AS universo_trauma")
    parts.append(f"sum(CASE WHEN {expr4} IN ({lst(INF4)}) OR substr({expr4},1,3) = 'J36' THEN 1 ELSE 0 END) AS infecciones_11")
    parts.append("count(*) AS egresos_totales")
    return parts

rows, grupo_rows, inf_rows, mech_rows = [], [], [], []
for y in deis_years:
    load_table(DEIS_DIR / DEIS_PATTERN.format(year=y), ";", DEIS_KEEP, name="deis")
    raw4 = "upper(trim(substr(DIAG1, 1, 4)))"
    clean4 = "substr(upper(regexp_replace(trim(DIAG1), '[^A-Za-z0-9]', '', 'g')), 1, 4)"
    for et, ex in [("tal_cual", raw4), ("sin_puntos", clean4)]:
        r = con.execute(f"SELECT {', '.join(def_counts(ex, et))} FROM deis").df()
        r.insert(0, "lectura_dx", et); r.insert(0, "anio", y); rows.append(r)
    ge = "GRUPO_EDAD" if has("GRUPO_EDAD", "deis") else "NULL"
    g = con.execute(f"""SELECT substr({clean4},1,3) AS dx3, {clean4} AS dx4, {ge} AS grupo_edad, count(*) AS n FROM deis
                        WHERE substr({clean4},1,3) IN ('Q35','Q36','Q37','Q75','Q87') GROUP BY 1,2,3""").df()
    g.insert(0, "anio", y); grupo_rows.append(g)
    i = con.execute(f"""SELECT CASE WHEN substr({clean4},1,3)='J36' THEN 'J36' ELSE {clean4} END AS codigo, count(*) AS n FROM deis
                        WHERE {clean4} IN ({lst(INF4)}) OR substr({clean4},1,3)='J36' GROUP BY 1""").df()
    i.insert(0, "anio", y); inf_rows.append(i)
    if has("DIAG2", "deis"):
        m = con.execute(f"""SELECT CASE WHEN DIAG2 IS NULL OR trim(DIAG2) IN ('', '*') THEN 'vacío'
                                        WHEN regexp_matches(upper(trim(DIAG2)), '^[V-Y][0-9]') THEN substr(upper(trim(DIAG2)),1,1)
                                        ELSE 'otro' END AS diag2_clase, count(*) AS n FROM deis
                            WHERE {clean4} IN ({lst(DEFS['trauma_C_facial'])}) GROUP BY 1""").df()
        m.insert(0, "anio", y); mech_rows.append(m)
    log(f"[{y}] definiciones contadas")

defs_deis = pd.concat(rows, ignore_index=True)
save(defs_deis, "deis_definiciones_candidatas")
save(pd.concat(grupo_rows, ignore_index=True), "deis_fisuras_craneo_por_grupo_edad")
save(pd.concat(inf_rows, ignore_index=True), "deis_infecciones_por_codigo")
if mech_rows: save(pd.concat(mech_rows, ignore_index=True), "deis_trauma_diag2_clase")
print(defs_deis[defs_deis["lectura_dx"] == "tal_cual"].drop(columns="lectura_dx").to_string(index=False))
dif = defs_deis.pivot_table(index="anio", columns="lectura_dx", values="trauma_A_blando_duro")
if (dif.get("tal_cual") != dif.get("sin_puntos")).any():
    log("ATENCIÓN: los conteos cambian al quitar puntos de DIAG1 (ver deis_definiciones_candidatas.csv)")

## 3. GRD: estructura, etiquetas y definiciones de tus notebooks

Reproduce, por año, el pool traumático y el componente facial del notebook 2019-2025 y las definiciones de
episodios del notebook de craneosinostosis, para compararlos con `control_cohorte_por_anio.csv` y
`episodios_por_definicion.csv`.

In [ ]:
GRD_CAT = ["TIPO_ACTIVIDAD", "TIPOALTA", "SEXO", "USOSPABELLON", "TRAMO_EDAD", "PREVISION", "IR_29301_SEVERIDAD", "TIPO_INGRESO"]
GRD_KEEP = set(GRD_CAT) | {"COD_HOSPITAL", "COMUNA", "FECHA_NACIMIENTO", "FECHA_INGRESO", "FECHAALTA", "FECHAINTERV1",
                           "IR_29301_COD_GRD", "IR_29301_PESO"}
GRD_PAT = (r"DIAGNOSTICO_?[0-9]+", r"PROCEDIMIENTO_?[0-9]+")
grd_years = [y for y in GRD_YEARS if (GRD_DIR / GRD_PATTERN.format(year=y)).exists()]
log(f"Años GRD encontrados: {grd_years}")

def numcols(prefix):
    cs = [r[0] for r in con.execute("DESCRIBE grd").fetchall() if re.fullmatch(prefix + r"_?[0-9]+", r[0])]
    return sorted(cs, key=lambda c: int(re.sub(r"\D", "", c)))

c4 = lambda c: f"substr(upper(regexp_replace(coalesce({qi(c)}, ''), '[^A-Za-z0-9]', '', 'g')), 1, 4)"
FACIAL = SOFT + HARD + OTHER; CRANIAL = ["S020", "S021"]
CS_PROCS = ["0201", "0203", "0204", "0205", "0206", "0125"]

g_info, g_cols, g_vc, g_fmt, g_def, g_cs = [], [], [], [], [], []
g_err = []
for y in grd_years:
    try:
        enc, allcols = load_table(GRD_DIR / GRD_PATTERN.format(year=y), "|", GRD_KEEP, GRD_PAT, name="grd")
    except Exception as e:  # noqa: BLE001  (un año con otro formato no detiene la sección)
        log(f"GRD {y}: {e}")
        g_err.append({"anio": y, "error": str(e)[:600]}); continue
    dx, pr = numcols("DIAGNOSTICO"), numcols("PROCEDIMIENTO")
    n = con.execute("SELECT count(*) FROM grd").fetchone()[0]
    g_info.append({"anio": y, "encoding": enc, "separador": LOAD_INFO["grd"]["separador"], "filas": n,
                   "n_columnas": len(allcols), "n_diagnosticos": len(dx), "n_procedimientos": len(pr)})
    g_cols += [{"anio": y, "columna": c} for c in allcols]
    for c in GRD_CAT + ["COD_HOSPITAL"]:
        if has(c, "grd"):
            g_vc.append(value_counts("grd", c, y).head(400))
    if has("COMUNA", "grd"):
        t = con.execute("""SELECT COMUNA AS valor, count(*) AS n FROM grd GROUP BY 1 ORDER BY 2 DESC LIMIT 500""").df()
        t.insert(0, "columna", "COMUNA"); t.insert(0, "anio", y); g_vc.append(t)
    for c in ["FECHA_NACIMIENTO", "FECHA_INGRESO", "FECHAALTA", "FECHAINTERV1", "IR_29301_PESO", "COMUNA"] + dx[:2] + pr[:1]:
        if has(c, "grd"):
            f = con.execute(f"SELECT {shape_expr(c)} AS forma, count(*) AS n FROM grd GROUP BY 1 ORDER BY 2 DESC LIMIT 10").df()
            f.insert(0, "columna", c); f.insert(0, "anio", y); g_fmt.append(f)

    # Pool traumático del notebook 2019-2025: DX1 S00-S99, T00-T14, T20-T32; hospitalización estricta
    d1 = c4(dx[0])
    act = "upper(strip_accents(trim(coalesce(TIPO_ACTIVIDAD,''))))" if has("TIPO_ACTIVIDAD", "grd") else "'HOSPITALIZACION'"
    univ = (f"(substr({d1},1,1)='S' AND try_cast(substr({d1},2,2) AS INTEGER) BETWEEN 0 AND 99) OR "
            f"(substr({d1},1,1)='T' AND (try_cast(substr({d1},2,2) AS INTEGER) BETWEEN 0 AND 14 OR try_cast(substr({d1},2,2) AS INTEGER) BETWEEN 20 AND 32))")
    any_in = lambda codes: "(" + " OR ".join(f"{c4(c)} IN ({lst(codes)})" for c in dx) + ")"
    other_inj = "(" + " OR ".join(f"(regexp_matches({c4(c)}, '^(S[0-9]{{2}}|T[0-7][0-9])') AND {c4(c)} NOT IN ({lst(FACIAL)}))" for c in dx) + ")"
    q = f"""
      SELECT {act} AS actividad,
             count(*) AS egresos,
             sum(CASE WHEN {univ} THEN 1 ELSE 0 END) AS pool_trauma_dx1,
             sum(CASE WHEN ({univ}) AND {any_in(FACIAL)} THEN 1 ELSE 0 END) AS facial_cualquier_posicion,
             sum(CASE WHEN ({univ}) AND {any_in(FACIAL)} AND NOT {other_inj} THEN 1 ELSE 0 END) AS facial_aislado,
             sum(CASE WHEN ({univ}) AND {any_in(CRANIAL)} THEN 1 ELSE 0 END) AS craneal_cualquier_posicion,
             sum(CASE WHEN ({univ}) AND {any_in(FACIAL + CRANIAL)} THEN 1 ELSE 0 END) AS cmft_amplio,
             sum(CASE WHEN {d1} IN ({lst(FACIAL)}) THEN 1 ELSE 0 END) AS facial_dx1,
             sum(CASE WHEN {d1} IN ({lst(INF4)}) OR substr({d1},1,3)='J36' THEN 1 ELSE 0 END) AS infecciones_dx1
      FROM grd GROUP BY 1 ORDER BY 2 DESC"""
    r = con.execute(q).df(); r.insert(0, "anio", y); g_def.append(r)

    # Craneosinostosis: episodios con Q75.0/Q75.1/Q75.8/Q75.9/Q87.0 en cualquier posición (solo conteos agregados)
    cs_any = any_in(["Q750", "Q751", "Q758", "Q759", "Q870"])
    cols_cs = [c for c in ["TIPO_ACTIVIDAD", "FECHA_NACIMIENTO", "TRAMO_EDAD", "FECHA_INGRESO", "USOSPABELLON", "FECHAINTERV1"] if has(c, "grd")]
    sel = ", ".join([f"{d1} AS dx1"] + [qi(c) for c in cols_cs] +
                    [f"{any_in(['Q750'])} AS q750_any",
                     ("(" + " OR ".join(f"{c4(c)} IN ({lst(CS_PROCS)})" for c in pr) + ")" if pr else "FALSE") + " AS proc_cs",
                     ("concat_ws(';', " + ", ".join(c4(c) for c in pr) + ")" if pr else "''") + " AS procs"])
    e = con.execute(f"SELECT {sel} FROM grd WHERE {cs_any}").df()
    if len(e):
        def pdate(s):
            s = s.astype("string").str.strip().str.slice(0, 10)
            best, ok = None, -1
            for fmt in ["%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y", "%Y/%m/%d", "%m/%d/%Y", "%Y%m%d"]:
                d = pd.to_datetime(s, format=fmt, errors="coerce"); k = d.notna().mean()
                if k > ok: best, ok = d, k
            return best
        ing = pdate(e["FECHA_INGRESO"]) if "FECHA_INGRESO" in e else None
        if "FECHA_NACIMIENTO" in e and ing is not None:
            meses = (ing - pdate(e["FECHA_NACIMIENTO"])).dt.days / 30.4375
            e["edad_0a9"] = (meses / 12).between(0, 9.999)
            e["fuente_edad"] = "fecha_nacimiento"
        else:
            e["edad_0a9"] = e.get("TRAMO_EDAD", pd.Series(index=e.index, dtype="string")).astype("string").fillna("")
            e["fuente_edad"] = "tramo"
        actn = e.get("TIPO_ACTIVIDAD", pd.Series("HOSPITALIZACION", index=e.index)).astype("string").map(
            lambda v: unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode().upper().strip())
        e["hosp"] = actn.eq("HOSPITALIZACION")
        pab = pd.to_numeric(e.get("USOSPABELLON"), errors="coerce").fillna(0).gt(0) if "USOSPABELLON" in e else False
        e["pabellon_o_proc"] = pab | e["proc_cs"]
        agg = (e.groupby(["dx1", "hosp", "fuente_edad", "edad_0a9", "q750_any", "proc_cs", "pabellon_o_proc"], dropna=False)
                .size().rename("n").reset_index())
        agg.insert(0, "anio", y); g_cs.append(agg)
        top = (e[e["dx1"].eq("Q750")]["procs"].str.split(";").explode().loc[lambda s: s.ne("")]
               .value_counts().head(25).rename_axis("procedimiento").reset_index(name="n"))
        top.insert(0, "anio", y); save(top, f"grd_craneo_procedimientos_top_{y}")
    log(f"[{y}] enc={enc} filas={n:,} diagnósticos={len(dx)} procedimientos={len(pr)}")

save(pd.DataFrame(g_info), "grd_info_archivos")
if g_err: save(pd.DataFrame(g_err), "grd_errores_lectura")
else: (OUT / "grd_errores_lectura.csv").unlink(missing_ok=True)   # no dejar errores de corridas anteriores
save(pd.DataFrame(g_cols), "grd_columnas")
for lst, nombre in [(g_vc, "grd_valores"), (g_fmt, "grd_formatos"), (g_cs, "grd_craneo_episodios")]:
    if lst: save(pd.concat(lst, ignore_index=True), nombre)
if g_def:
    grd_defs = pd.concat(g_def, ignore_index=True); save(grd_defs, "grd_definiciones_candidatas")
    print(grd_defs[grd_defs["actividad"] == "HOSPITALIZACION"].to_string(index=False))
if g_err:
    print("Años con error de lectura (ver grd_errores_lectura.csv):", [e["anio"] for e in g_err])

### 3b. DEIS: columna `PROCED` (2001-2020)

Solo la forma de los valores y los 20 más frecuentes en cuatro años, para saber si sirve como procedimiento.

In [ ]:
p_rows = []
for y in [2001, 2010, 2019, 2020]:
    f = DEIS_DIR / DEIS_PATTERN.format(year=y)
    if not f.exists():
        continue
    try:
        load_table(f, ";", {"PROCED", "INTERV_Q", "DIAG1"}, name="dp")
    except Exception as e:  # noqa: BLE001
        log(f"DEIS {y}: {e}"); continue
    if not has("PROCED", "dp"):
        continue
    forma = con.execute(f"SELECT {shape_expr('PROCED')} AS forma, count(*) AS n FROM dp GROUP BY 1 ORDER BY 2 DESC LIMIT 10").df()
    forma.insert(0, "tipo", "forma"); forma.insert(0, "anio", y); p_rows.append(forma.rename(columns={"forma": "valor"}))
    top = con.execute("SELECT PROCED AS valor, count(*) AS n FROM dp GROUP BY 1 ORDER BY 2 DESC LIMIT 20").df()
    top.insert(0, "tipo", "top"); top.insert(0, "anio", y); p_rows.append(top)
    cruce = con.execute("SELECT INTERV_Q AS valor, count(*) FILTER (WHERE PROCED IS NOT NULL AND trim(PROCED) NOT IN ('', '*')) AS n "
                        "FROM dp GROUP BY 1").df() if has("INTERV_Q", "dp") else pd.DataFrame()
    if len(cruce): cruce.insert(0, "tipo", "proced_no_vacio_por_INTERV_Q"); cruce.insert(0, "anio", y); p_rows.append(cruce)
if p_rows:
    save(pd.concat(p_rows, ignore_index=True), "deis_proced")

## 4. INE, nacidos vivos, IPC y productividad: estructura

In [ ]:
def describe_excel(path, nombre, **kw):
    if not Path(path).exists():
        log(f"{nombre}: no encontrado ({path})"); return None
    x = pd.read_excel(path, **kw)
    x.columns = [" ".join(str(c).split()) for c in x.columns]
    info = pd.DataFrame({"columna": x.columns, "tipo": [str(t) for t in x.dtypes],
                         "n_distintos": [x[c].nunique() for c in x.columns],
                         "ejemplos": [", ".join(map(str, x[c].dropna().unique()[:8])) for c in x.columns]})
    save(info, f"{nombre}_estructura"); return x

ine = describe_excel(INE_FILE, "ine_comunal", sheet_name=0)
if ine is not None:
    up = {c: norm_col(c) for c in ine.columns}
    pop = [c for c, u in up.items() if re.fullmatch(r"(POBLACION_)?(19|20)\d{2}", u)]
    area = next((c for c, u in up.items() if u.startswith("AREA")), None)
    edad = next((c for c, u in up.items() if u in ("EDAD", "GRUPO_EDAD") or u.startswith("EDAD")), None)
    res = {"columnas_poblacion": [pop[0], pop[-1]] if pop else None,
           "valores_area": ine[area].astype(str).value_counts().to_dict() if area else None,
           "edades": sorted(map(str, ine[edad].unique()))[:120] if edad else None,
           "filas": len(ine)}
    tot = ine[pop].apply(pd.to_numeric, errors="coerce").sum()
    res["total_sumando_todas_las_filas"] = {k: round(float(v)) for k, v in tot.items()}
    json.dump(res, open(OUT / "ine_comunal_resumen.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
    print(json.dumps({k: res[k] for k in ["columnas_poblacion", "valores_area", "filas"]}, ensure_ascii=False, default=str))

for path, nombre in [(INE_REGION_FILE, "ine_regional"), (DENOM_FLP_FILE, "denominador_flp")]:
    if Path(path).exists():
        xl = pd.ExcelFile(path)
        save(pd.DataFrame({"hoja": xl.sheet_names}), f"{nombre}_hojas")
        describe_excel(path, nombre, sheet_name=0, nrows=60)

for path, nombre in [(BIRTHS_REGION_FILE, "nacimientos_region"), (BIRTHS_FILE, "nacimientos_chile"), (PROD_FILE, "productividad")]:
    if Path(path).exists():
        x = pd.read_csv(path, sep=None, engine="python")
        save(pd.DataFrame({"columna": x.columns, "ejemplos": [", ".join(map(str, x[c].unique()[:10])) for c in x.columns]}), f"{nombre}_estructura")
        if nombre.startswith("nacimientos"):
            save(x, nombre)   # tabla agregada (año, región, nacidos vivos)
    else:
        log(f"{nombre}: no encontrado ({path})")

if Path(CPI_FILE).exists():
    cpi = pd.read_excel(CPI_FILE, header=None, nrows=12)
    save(cpi.astype(str), "ipc_primeras_filas")

## 5. *(Opcional)* Correr el pipeline del observatorio con tus archivos

Con `RUN_PIPELINE = True`, copia el repositorio, registra tus archivos locales (rutas absolutas, no se suben a
ningún lado) y ejecuta `pipeline/run_pipeline.R`. Exporta solo series nacionales y regionales por módulo,
subgrupo y año, y el informe de validación. Puede tardar; los años ya procesados quedan en caché.

In [ ]:
def find_rscript():
    if RSCRIPT and Path(RSCRIPT).exists(): return str(RSCRIPT)
    w = shutil.which("Rscript")
    if w: return w
    for base in [Path(r"C:/Program Files/R"), Path.home() / "AppData/Local/Programs/R"]:
        if base.exists():
            c = sorted(base.glob("R-*/bin/Rscript.exe"))
            if c: return str(c[-1])
    return None

def yml_list(versions):
    out = []
    for v in versions:
        out.append(f"      - version: '{v['version']}'\n        anios: [{v['anio']}]\n        archivo: '{v['archivo']}'")
    return "\n".join(out) if out else ""

if RUN_PIPELINE:
    import subprocess, sys
    rs = find_rscript(); assert rs, "No encontré Rscript: define RSCRIPT"
    work = OUT.parent / "observatorio_corrida_local"
    if not work.exists():
        shutil.copytree(REPO_DIR, work, ignore=shutil.ignore_patterns("data-raw", "data-cache", ".git"))
    else:  # actualiza el código, conserva la caché
        for sub in ["R", "pipeline", "config", "dictionaries"]:
            shutil.copytree(REPO_DIR / sub, work / sub, dirs_exist_ok=True)
    P = lambda p: str(p).replace("\\", "/")
    deis_v = [{"version": str(y), "anio": y, "archivo": P(DEIS_DIR / DEIS_PATTERN.format(year=y))}
              for y in PIPE_DEIS_YEARS if (DEIS_DIR / DEIS_PATTERN.format(year=y)).exists()]
    grd_v = [{"version": str(y), "anio": y, "archivo": P(GRD_DIR / GRD_PATTERN.format(year=y))}
             for y in PIPE_GRD_YEARS if (GRD_DIR / GRD_PATTERN.format(year=y)).exists()]
    src = (work / "config" / "sources.yml").read_text(encoding="utf-8")
    def put(text, key, versions_yml, extra=""):
        pat = rf"(  {key}:\n(?:    .*\n)*?)    versiones: \[\]"
        return re.sub(pat, lambda m: m.group(1) + "    versiones:\n" + versions_yml, text, count=1)
    src = put(src, "egresos_deis", yml_list(deis_v))
    src = put(src, "grd", yml_list(grd_v))
    one = lambda p: yml_list([{"version": "local", "anio": "", "archivo": P(p)}])
    if INE_FILE.exists(): src = put(src, "poblacion_ine", one(INE_FILE))
    if BIRTHS_REGION_FILE.exists(): src = put(src, "nacimientos_region", one(BIRTHS_REGION_FILE))
    if CPI_FILE.exists(): src = put(src, "ipc", one(CPI_FILE))
    (work / "config" / "sources.yml").write_text(src, encoding="utf-8")
    env = {**__import__("os").environ, "LANG": "C.UTF-8", "LC_ALL": "C.UTF-8"}
    log("Ejecutando el pipeline (puede tardar)…")
    p = subprocess.run([rs, "pipeline/run_pipeline.R"], cwd=work, capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    (OUT / "pipeline_log.txt").write_text(p.stdout + "\n" + p.stderr, encoding="utf-8")
    log(p.stdout[-3000:]); log(p.stderr[-3000:])
    ext = work / "inst" / "extdata"
    if (ext / "cubo.csv.gz").exists():
        cubo = pd.read_csv(ext / "cubo.csv.gz", dtype={"subgrupo": str})
        save(cubo.groupby(["fuente", "modulo", "subgrupo", "anio"], dropna=False)[["n", "fallecidos", "n_pabellon", "n_aislado", "valor_grd", "n_con_valor"]].sum(min_count=1).reset_index(), "pipeline_nacional")
        save(cubo.groupby(["fuente", "modulo", "subgrupo", "region"], dropna=False)["n"].sum().reset_index(), "pipeline_regional")
        save(cubo.groupby(["fuente", "modulo", "grupo_edad_5"], dropna=False)["n"].sum().reset_index(), "pipeline_grupo_quinquenal")
        if (ext / "universo.csv.gz").exists():
            u = pd.read_csv(ext / "universo.csv.gz")
            save(u.groupby(["fuente", "modulo", "anio"])["n_universo"].sum().reset_index(), "pipeline_universo")
        for f in ["validacion.json", "manifest.json"]:
            if (ext / f).exists(): shutil.copy(ext / f, OUT / f"pipeline_{f}")
else:
    log("Sección 5 omitida (RUN_PIPELINE = False)")

## 6. Preguntas para cuadrar el método con las publicaciones

Responde directamente en la celda de abajo: edita los textos entre comillas. Se guardan en `respuestas.json`
dentro del zip. Si alguna cifra publicada está en una tabla, basta con anotar el valor y la tabla.

In [ ]:
RESPUESTAS = {
    # Trauma (DEIS, "Dual approach…")
    "trauma_script_publicado": "¿Este_si.R (sin S02.0) o final_final.R (con S02.0 en 'duro')? ",
    "trauma_denominador_publicado": "¿Por 1.000 egresos por trauma (S00-S99 + T00-T14), por población, o ambos? ",
    "trauma_años_publicados": "¿2001-2024? ",
    "trauma_cifra_control": "Una cifra publicada para cuadrar, p. ej. total de egresos CMF 2001-2024 o de un año: ",
    "trauma_ITS": "¿Quiebres COVID 2020-2021 y post 2022, NB con offset log(egresos trauma), EF región/sexo/edad/mecanismo? ",
    "trauma_mecanismo": "¿Los 6 grupos de final_final.R (ext_cause_6grp) o los 7 del notebook 2019-2025? ",
    # Trauma GRD 2019-2025
    "trauma_grd_estado": "¿Publicado o en revisión? ¿Qué cifra o tabla uso para cuadrar? ",
    # Infecciones
    "infecciones_estado": "¿Publicado? ¿Revista y años (2002-2025)? ",
    "infecciones_grupos_edad": "¿Grupos 0-14, 15-64, ≥65 del libro INE regional? ",
    "infecciones_cifra_control": "",
    # Fisuras
    "fisuras_estado": "¿Publicado? ¿Denominador principal: INE 0 años calibrado a nacidos vivos (2002-2023)? ",
    "fisuras_cifra_control": "",
    # Craneosinostosis
    "craneo_serie_deis": "El notebook 'craneosinostosis_series' (DEIS 2002-2025) no me llegó: ¿puedes adjuntarlo? ¿Denominador: población 0-9 o nacidos vivos? ",
    "craneo_estado": "",
    # Dotación
    "oferta_csv": "¿Puedes exportar desde el notebook de dotación una tabla por Servicio de Salud (indicadores finales)? ",
    # General
    "nombre_publico": "¿'Observatorio CMF Chile' u 'Observatorio de Cirugía Maxilofacial de Chile'? ",
    "otros_comentarios": "",
}
json.dump(RESPUESTAS, open(OUT / "respuestas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## 7. Empaquetar

In [ ]:
prev = OUT / "salidas_previas"; prev.mkdir(exist_ok=True)
for f in OUTPUTS_PREVIOS:
    if Path(f).exists():
        shutil.copy(f, prev / Path(f).name); log(f"Copiado {Path(f).name}")
(OUT / "log.txt").write_text("\n".join(LOG), encoding="utf-8")
zp = OUT.parent / f"diagnostico_observatorio_{datetime.date.today():%Y%m%d}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file(): z.write(f, f.relative_to(OUT.parent))
print(f"Listo. Mándame este archivo: {zp}  ({zp.stat().st_size/1e6:.1f} MB)")